# AI Agents: Overview, Frameworks, and Design Patterns

A comprehensive session covering:
1. **What is an Agent?** — definition, anatomy, and how they differ from plain LLM calls
2. **Agent Framework Overview** — the landscape of tools and libraries
3. **Workflow & Agent Design Patterns** — Anthropic's recommended patterns, each with a Gradio UI
4. **Model Context Protocol (MCP)** — what it is, building a server, wiring an OpenAI agent to it

## Setup

In [ ]:
import os
import json
import time
import concurrent.futures
from collections import Counter
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
client = OpenAI()

MODEL = "gpt-4.1-mini"

# Shared helper used by all patterns
def llm_call(system: str, user: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}]
    )
    return response.choices[0].message.content

print("Setup complete. Model:", MODEL)

---
# Part 1 — What is an Agent?

## 1.1 Plain LLM call vs. an Agent

| | Plain LLM Call | Agent |
|---|---|---|
| **Interaction** | Single request → single response | Multi-step, iterative loop |
| **Memory** | None (stateless) | Can read/write state across turns |
| **Tools** | None | Can call external functions/APIs |
| **Decision-making** | No | Decides *what to do next* based on intermediate results |
| **Goal** | Answer a question | Accomplish a task |

## 1.2 Anatomy of an Agent

```
┌────────────────────────────────────┐
│            AGENT LOOP              │
│                                    │
│  Input/Goal                        │
│      │                             │
│      ▼                             │
│  ┌─────────┐   Tool calls    ┌───┐ │
│  │   LLM   │ ──────────────► │   │ │
│  │  Brain  │ ◄────────────── │ T │ │
│  └─────────┘   Tool results  │ O │ │
│      │                       │ O │ │
│      │ (stop condition met?) │ L │ │
│      ▼                       │ S │ │
│  Final Answer                └───┘ │
└────────────────────────────────────┘
```

**Key components:**
- **LLM (Brain)** — reasons about the goal and decides the next action
- **Tools** — functions the LLM can invoke (search, DB query, API call, code execution)
- **Memory** — short-term (conversation context) and long-term (vector DB, files)
- **Stop condition** — when the agent decides it has finished

## 1.3 Simple Agent Example — Calculator Agent

The agent has three math tools (`add`, `multiply`, `power`) and loops until it produces a final answer.

In [ ]:
# ── Tool definitions ─────────────────────────────────────────────────────────
def add(a: float, b: float) -> float:       return a + b
def multiply(a: float, b: float) -> float:  return a * b
def power(base: float, exp: float) -> float: return base ** exp

CALC_TOOL_MAP = {"add": add, "multiply": multiply, "power": power}

calc_tools = [
    {"type": "function", "function": {
        "name": "add", "description": "Add two numbers",
        "parameters": {"type": "object", "properties": {
            "a": {"type": "number"}, "b": {"type": "number"}},
            "required": ["a", "b"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "multiply", "description": "Multiply two numbers",
        "parameters": {"type": "object", "properties": {
            "a": {"type": "number"}, "b": {"type": "number"}},
            "required": ["a", "b"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "power", "description": "Raise base to exponent",
        "parameters": {"type": "object", "properties": {
            "base": {"type": "number"}, "exp": {"type": "number"}},
            "required": ["base", "exp"], "additionalProperties": False}}},
]

# ── Agent loop ───────────────────────────────────────────────────────────────
def run_calculator_agent(user_query: str) -> str:
    messages = [
        {"role": "system", "content": "You are a calculator assistant. Use the tools to solve math problems step by step."},
        {"role": "user",   "content": user_query},
    ]
    trace_lines = [f"**Goal:** {user_query}\n"]

    for step in range(1, 8):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=calc_tools)
        msg    = response.choices[0].message
        finish = response.choices[0].finish_reason

        if finish == "stop":
            trace_lines.append(f"**Step {step}:** Agent finished.")
            trace_lines.append(f"\n---\n**Answer:** {msg.content}")
            return "\n".join(trace_lines)

        if finish == "tool_calls":
            messages.append(msg)
            for tc in msg.tool_calls:
                fn   = tc.function.name
                args = json.loads(tc.function.arguments)
                result = CALC_TOOL_MAP[fn](**args)
                trace_lines.append(f"**Step {step}:** Called `{fn}({args})` → `{result}`")
                messages.append({"role": "tool", "content": str(result), "tool_call_id": tc.id})

    return "\n".join(trace_lines) + "\n\nMax steps reached."

# ── Gradio UI ────────────────────────────────────────────────────────────────
with gr.Blocks(title="Calculator Agent") as calculator_demo:
    gr.Markdown("## Calculator Agent\nAsk any multi-step math question — the agent picks and chains tools automatically.")
    with gr.Row():
        with gr.Column():
            calc_input = gr.Textbox(
                label="Math Question",
                placeholder="e.g. What is (3 + 7) raised to the power of 2, then multiplied by 5?",
                lines=2
            )
            calc_btn = gr.Button("Run Agent", variant="primary")
        with gr.Column():
            calc_output = gr.Markdown(label="Agent Trace & Answer")

    calc_btn.click(fn=run_calculator_agent, inputs=calc_input, outputs=calc_output)
    gr.Examples(
        examples=[
            ["What is (3 + 7) raised to the power of 2, then multiplied by 5?"],
            ["Add 15 and 25, then raise the result to the power of 3"],
            ["Multiply 6 by 7, then add 100 to that result"],
        ],
        inputs=calc_input
    )

calculator_demo.launch()

---
# Part 2 — Agent Framework Overview

## Why use a framework?

Building the agent loop manually works but quickly becomes complex:
- Managing conversation history
- Routing tool calls
- Handling retries, errors, streaming
- Composing multiple agents together

## 2.1 Framework Landscape

| Framework | By | Key Strengths |
|---|---|---|
| **OpenAI Agents SDK** | OpenAI | Handoffs, guardrails, tracing — first-class multi-agent |
| **LangChain / LangGraph** | LangChain | Huge ecosystem, graph-based workflows, many integrations |
| **LlamaIndex** | LlamaIndex | RAG-first, strong data connectors |
| **AutoGen** | Microsoft | Conversational multi-agent, code execution |
| **CrewAI** | CrewAI | Role-based agent crews, easy to read YAML config |
| **Semantic Kernel** | Microsoft | .NET + Python, enterprise integrations |
| **Claude Agent SDK** | Anthropic | Built for Claude, MCP protocol support |

## 2.2 Multi-Agent Architecture Demo

An **orchestrator** classifies the question and routes it to a specialist agent (Math / History / Science).

In [ ]:
# ── Multi-agent triage system ────────────────────────────────────────────────
SPECIALIST_AGENTS = {
    "MATH":    ("Math Expert",    "You are a mathematics expert. Give precise, step-by-step solutions."),
    "HISTORY": ("History Expert", "You are a history expert. Give accurate, context-rich answers."),
    "SCIENCE": ("Science Expert", "You are a science expert. Give clear, evidence-based explanations."),
    "GENERAL": ("General Assistant", "You are a helpful, knowledgeable assistant."),
}

def run_triage_agent(question: str) -> tuple[str, str]:
    category = llm_call(
        system="Classify this question into exactly one of: MATH, HISTORY, SCIENCE, GENERAL. Reply with ONLY the category word.",
        user=question
    ).strip().upper()
    if category not in SPECIALIST_AGENTS:
        category = "GENERAL"

    role, instructions = SPECIALIST_AGENTS[category]
    answer = llm_call(system=instructions, user=question)
    return f"Routed to: **{role}** (`{category}`)", answer

# ── Gradio UI ────────────────────────────────────────────────────────────────
with gr.Blocks(title="Multi-Agent Triage") as triage_demo:
    gr.Markdown("## Multi-Agent Triage System\nThe orchestrator classifies your question and routes it to the right specialist agent.")
    with gr.Row():
        with gr.Column():
            triage_input = gr.Textbox(label="Your Question", placeholder="Ask anything...", lines=2)
            triage_btn   = gr.Button("Ask", variant="primary")
        with gr.Column():
            triage_route  = gr.Markdown(label="Routing Decision")
            triage_answer = gr.Markdown(label="Answer")

    triage_btn.click(fn=run_triage_agent, inputs=triage_input, outputs=[triage_route, triage_answer])
    gr.Examples(
        examples=[
            ["What is the integral of x squared?"],
            ["When did the Roman Empire fall and what caused it?"],
            ["Why does ice float on water?"],
            ["What are the best practices for writing clean code?"],
        ],
        inputs=triage_input
    )

triage_demo.launch()

---
# Part 3 — Workflow & Agent Design Patterns

> Source: [Anthropic — Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents)

Anthropic organises LLM system designs into two categories:

- **Workflows** — the developer defines the control flow; the LLM fills in the content at each fixed step
- **Agents** — the LLM itself decides the control flow dynamically at runtime

```
Workflows (developer controls the flow)    Agents (LLM controls the flow)
────────────────────────────────────────   ──────────────────────────────
1. Prompt Chaining                         6. Autonomous Agent
2. Routing
3. Parallelization
4. Orchestrator-Workers
5. Evaluator-Optimizer
```

Patterns 1–5 are covered first. A detailed **Workflows vs. Agents** comparison appears between Pattern 5 and Pattern 6.

---
## Pattern 1 — Prompt Chaining

**Concept:** Break a task into a fixed sequence of LLM calls where each step feeds the next.

```
Input → [Step 1: Outline] → [Step 2: Draft] → [Step 3: Polish] → Output
```

**When to use:** Task can be decomposed into clear sequential sub-steps; intermediate outputs can be validated.

In [ ]:
# ── Prompt Chaining: Blog Post Pipeline ──────────────────────────────────────
def prompt_chaining_pipeline(topic: str) -> tuple[str, str, str]:
    outline = llm_call(
        system="You are a technical blog writer. Create concise outlines.",
        user=f"Create a 4-point outline for a blog post titled: '{topic}'"
    )
    draft_intro = llm_call(
        system="You are a technical blog writer. Write engaging introductions.",
        user=f"Write a 2-paragraph introduction based on this outline:\n{outline}"
    )
    polished = llm_call(
        system="You are an editor. Make text more engaging, punchy, and professional.",
        user=f"Polish and improve this introduction:\n{draft_intro}"
    )
    return outline, draft_intro, polished

# ── Gradio UI ────────────────────────────────────────────────────────────────
with gr.Blocks(title="Prompt Chaining") as chaining_demo:
    gr.Markdown("## Pattern 1 — Prompt Chaining\nA blog post is created through 3 chained LLM calls: Outline → Draft → Polish.")
    with gr.Row():
        chain_input = gr.Textbox(label="Blog Post Topic", placeholder="e.g. Why AI Agents are the next shift in software", lines=2)
        chain_btn   = gr.Button("Generate", variant="primary")
    with gr.Row():
        chain_outline  = gr.Textbox(label="Step 1 — Outline",       lines=6, interactive=False)
        chain_draft    = gr.Textbox(label="Step 2 — Draft Intro",   lines=6, interactive=False)
        chain_polished = gr.Textbox(label="Step 3 — Polished Intro",lines=6, interactive=False)

    chain_btn.click(fn=prompt_chaining_pipeline, inputs=chain_input,
                    outputs=[chain_outline, chain_draft, chain_polished])
    gr.Examples(
        examples=[
            ["Why AI Agents are the next big shift in software development"],
            ["The future of remote work in a post-AI world"],
            ["How vector databases are changing the way we build AI apps"],
        ],
        inputs=chain_input
    )

chaining_demo.launch()

---
## Pattern 2 — Routing

**Concept:** Classify the input first, then route to the most appropriate specialized handler.

```
             ┌──► [Billing Specialist]
Input ──► [Classifier] ──► [Technical Support]
             └──► [General Service]
```

**When to use:** Inputs fall into distinct categories each needing different instructions or models.

In [ ]:
# ── Routing Pattern: Customer Support ────────────────────────────────────────
SUPPORT_ROUTES = {
    "BILLING": "You are a billing specialist. Help with invoices, payments, refunds, and subscriptions. Be empathetic and solution-focused.",
    "TECHNICAL": "You are a technical support engineer. Help debug issues, configure settings, and explain error messages. Be precise.",
    "GENERAL": "You are a friendly customer service agent. Handle general inquiries and ensure customer satisfaction.",
}

def route_and_respond(message: str) -> tuple[str, str]:
    category = llm_call(
        system="Classify customer messages into exactly one of: BILLING, TECHNICAL, GENERAL. Reply with ONLY the category word.",
        user=message
    ).strip().upper()
    if category not in SUPPORT_ROUTES:
        category = "GENERAL"
    response = llm_call(system=SUPPORT_ROUTES[category], user=message)
    return f"**Routed to:** {category}", response

# ── Gradio UI ────────────────────────────────────────────────────────────────
with gr.Blocks(title="Routing Pattern") as routing_demo:
    gr.Markdown("## Pattern 2 — Routing\nYour message is classified and routed to the right specialist (Billing / Technical / General).")
    with gr.Row():
        with gr.Column():
            route_input = gr.Textbox(label="Customer Message", placeholder="Describe your issue...", lines=3)
            route_btn   = gr.Button("Submit", variant="primary")
        with gr.Column():
            route_label    = gr.Markdown(label="Routing Decision")
            route_response = gr.Textbox(label="Support Response", lines=8, interactive=False)

    route_btn.click(fn=route_and_respond, inputs=route_input, outputs=[route_label, route_response])
    gr.Examples(
        examples=[
            ["I was charged twice for my subscription this month!"],
            ["My API keeps returning a 429 error. How do I fix rate limiting?"],
            ["What are your support hours and how do I contact a human?"],
        ],
        inputs=route_input
    )

routing_demo.launch()

---
## Pattern 3 — Parallelization

**Concept:** Run multiple LLM calls simultaneously. Two variants:
- **Sectioning** — divide into independent sub-tasks, run in parallel
- **Voting** — run the same task N times, take majority result

```
         ┌──► [Worker 1: Health Benefits] ──┐
Input ───├──► [Worker 2: Value for Money] ──┼──► Aggregated Report
         ├──► [Worker 3: Target Audience] ──┤
         └──► [Worker 4: Key Risks]  ───────┘
```

In [ ]:
# ── Parallelization: Product Analyzer ────────────────────────────────────────
ANALYSIS_ASPECTS = {
    "Health & Wellness Benefits":   "Analyze only the health and wellness benefits of this product in 2-3 sentences.",
    "Value for Money":              "Analyze only the pricing and value proposition of this product in 2-3 sentences.",
    "Target Audience":              "Identify only the ideal target customer for this product in 2-3 sentences.",
    "Key Risks / Downsides":        "Identify the top 2 potential downsides or risks of this product.",
}

def analyze_aspect(aspect: str, prompt: str, product: str) -> tuple[str, str]:
    result = llm_call(
        system="You are a product analyst. Be concise and specific.",
        user=f"{prompt}\n\nProduct description:\n{product}"
    )
    return aspect, result

def parallel_product_analysis(product_desc: str) -> tuple[str, str, str, str, str]:
    start = time.time()
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = {
            executor.submit(analyze_aspect, aspect, prompt, product_desc): aspect
            for aspect, prompt in ANALYSIS_ASPECTS.items()
        }
        results = {}
        for f in concurrent.futures.as_completed(futures):
            aspect, result = f.result()
            results[aspect] = result
    elapsed = time.time() - start
    timing  = f"Completed **{len(results)} analyses in parallel** in `{elapsed:.1f}s`"
    return (timing,
            results.get("Health & Wellness Benefits", ""),
            results.get("Value for Money", ""),
            results.get("Target Audience", ""),
            results.get("Key Risks / Downsides", ""))

# ── Gradio UI ────────────────────────────────────────────────────────────────
with gr.Blocks(title="Parallelization Pattern") as parallel_demo:
    gr.Markdown("## Pattern 3 — Parallelization\nFour LLM workers analyze your product simultaneously — much faster than sequential calls.")
    with gr.Row():
        para_input = gr.Textbox(label="Product Description", placeholder="Describe a product...", lines=4)
        para_btn   = gr.Button("Analyze in Parallel", variant="primary")
    para_timing = gr.Markdown()
    with gr.Row():
        para_health   = gr.Textbox(label="Health & Wellness",  lines=4, interactive=False)
        para_value    = gr.Textbox(label="Value for Money",    lines=4, interactive=False)
    with gr.Row():
        para_audience = gr.Textbox(label="Target Audience",   lines=4, interactive=False)
        para_risks    = gr.Textbox(label="Key Risks",         lines=4, interactive=False)

    para_btn.click(
        fn=parallel_product_analysis, inputs=para_input,
        outputs=[para_timing, para_health, para_value, para_audience, para_risks]
    )
    gr.Examples(
        examples=[
            ["SmartDesk Pro: An AI-powered standing desk with posture monitoring, calendar sync, and height adjustment. Costs $1,299. Assembly required."],
            ["FitRing X: A smart ring that tracks sleep, heart rate, and stress levels. Waterproof, 7-day battery, costs $299."],
        ],
        inputs=para_input
    )

parallel_demo.launch()

---
## Pattern 4 — Orchestrator-Workers

**Concept:** A central **orchestrator** LLM dynamically plans the work and delegates to **worker** LLMs. Unlike prompt chaining, the orchestrator decides *at runtime* what to delegate.

```
                    ┌──► [Worker: Research sub-topic 1]
User ──► [Orchestrator] plans ──► [Worker: Research sub-topic 2]
                    └──► [Worker: Research sub-topic 3]
                              ↓
                    [Worker: Critic]  ──► [Worker: Formatter] ──► Final Report
```

In [ ]:
# ── Orchestrator-Workers: Research Report ────────────────────────────────────
def researcher_worker(topic: str) -> tuple[str, str]:
    return topic, llm_call(
        system="You are a research analyst. Provide 3 key facts with data points.",
        user=f"Research and provide 3 key facts about: {topic}"
    )

def critic_worker(content: str) -> str:
    return llm_call(
        system="You are a critical analyst. Identify gaps and limitations.",
        user=f"Identify 2 limitations or counterpoints for:\n{content}"
    )

def formatter_worker(content: str) -> str:
    return llm_call(
        system="You are a technical writer. Format content as a clear executive summary with markdown headers.",
        user=f"Format this as a concise executive summary:\n{content}"
    )

def run_orchestrator(research_topic: str) -> tuple[str, str]:
    # Orchestrator plans sub-topics
    plan_raw = llm_call(
        system="You manage a research pipeline. Output ONLY a JSON array of 3 specific sub-topic strings.",
        user=f"Break into 3 focused sub-topics: {research_topic}"
    )
    try:
        text = plan_raw.strip()
        if "```" in text:
            text = text.split("```")[1].lstrip("json")
        sub_topics = json.loads(text)
    except Exception:
        sub_topics = [research_topic + " overview",
                      research_topic + " challenges",
                      research_topic + " future"]

    plan_display = "**Orchestrator plan:**\n" + "\n".join(f"- {t}" for t in sub_topics)

    # Dispatch workers in parallel
    with concurrent.futures.ThreadPoolExecutor() as executor:
        research_results = dict(executor.map(researcher_worker, sub_topics))

    combined = "\n\n".join(f"**{t}:**\n{r}" for t, r in research_results.items())
    critique  = critic_worker(combined)
    report    = formatter_worker(combined + f"\n\n**Limitations:**\n{critique}")
    return plan_display, report

# ── Gradio UI ────────────────────────────────────────────────────────────────
with gr.Blocks(title="Orchestrator-Workers") as orchestrator_demo:
    gr.Markdown("## Pattern 4 — Orchestrator-Workers\nThe orchestrator dynamically plans sub-topics, dispatches parallel research workers, then critiques and formats the final report.")
    with gr.Row():
        orch_input = gr.Textbox(label="Research Topic", placeholder="e.g. The impact of AI Agents on software development", lines=2)
        orch_btn   = gr.Button("Generate Report", variant="primary")
    orch_plan   = gr.Markdown(label="Orchestrator Plan")
    orch_report = gr.Markdown(label="Final Report")

    orch_btn.click(fn=run_orchestrator, inputs=orch_input, outputs=[orch_plan, orch_report])
    gr.Examples(
        examples=[
            ["The impact of AI Agents on software development workflows"],
            ["Challenges and opportunities of renewable energy adoption"],
            ["How large language models are transforming education"],
        ],
        inputs=orch_input
    )

orchestrator_demo.launch()

---
## Pattern 5 — Evaluator-Optimizer

**Concept:** Two LLMs in a loop — a **generator** produces output, an **evaluator** critiques it, the generator improves.

```
                 ┌────────────── feedback ──────────────┐
                 ▼                                      │
Input ──► [Generator] ──► output ──► [Evaluator] ──► score
                                          │
                                   (score >= 8?)
                                          │ YES
                                          ▼
                                       Output
```

**When to use:** Quality matters and you have clear, articulable criteria for "good".

In [ ]:
# ── Evaluator-Optimizer: Code Review Loop ────────────────────────────────────
def code_generator(task: str, requirements: str, feedback: str = "") -> str:
    prompt = f"Task: {task}\n\nRequirements:\n{requirements}"
    if feedback:
        prompt += f"\n\nPrevious reviewer feedback to address:\n{feedback}"
    return llm_call(
        system="You are a Python developer. Write clean, production-quality Python code. Return ONLY the code.",
        user=prompt
    )

def code_evaluator(code: str, requirements: str) -> dict:
    raw = llm_call(
        system=(
            'You are a senior code reviewer. Reply ONLY in this JSON format: '
            '{"score": <1-10>, "passed": <true if score>=8>, "feedback": "<improvements>"}'
        ),
        user=f"Requirements:\n{requirements}\n\nCode:\n{code}"
    )
    try:
        text = raw.strip()
        if "```" in text:
            text = text.split("```")[1].lstrip("json")
        return json.loads(text)
    except Exception:
        return {"score": 5, "passed": False, "feedback": raw}

def run_evaluator_optimizer(task: str, requirements: str, max_iter: int = 3) -> tuple[str, str]:
    feedback   = ""
    trace_lines = []
    final_code  = ""

    for i in range(1, max_iter + 1):
        code  = code_generator(task, requirements, feedback)
        eval_ = code_evaluator(code, requirements)
        score   = eval_.get("score", 0)
        passed  = eval_.get("passed", False)
        feedback = eval_.get("feedback", "")
        final_code = code

        status = "PASSED" if passed else "needs improvement"
        trace_lines.append(f"**Iteration {i}:** Score `{score}/10` — {status}")
        if feedback and not passed:
            trace_lines.append(f"  - Feedback: {feedback[:200]}")

        if passed:
            trace_lines.append(f"\nAccepted after **{i}** iteration(s).")
            break
    else:
        trace_lines.append("\nMax iterations reached — using final version.")

    return "\n".join(trace_lines), final_code

# ── Gradio UI ────────────────────────────────────────────────────────────────
with gr.Blocks(title="Evaluator-Optimizer") as evaluator_demo:
    gr.Markdown("## Pattern 5 — Evaluator-Optimizer\nA generator writes code; an evaluator scores it and provides feedback; the generator iterates until the score is high enough.")
    with gr.Row():
        with gr.Column():
            eval_task  = gr.Textbox(label="Coding Task", lines=2,
                placeholder="e.g. Write a function to find the top N most frequent words in a string")
            eval_reqs  = gr.Textbox(label="Requirements / Acceptance Criteria", lines=5,
                placeholder="- Function signature: ...\n- Edge cases: ...")
            eval_iter  = gr.Slider(label="Max Iterations", minimum=1, maximum=5, step=1, value=3)
            eval_btn   = gr.Button("Run Loop", variant="primary")
        with gr.Column():
            eval_trace = gr.Markdown(label="Iteration Trace")
            eval_code  = gr.Code(label="Final Generated Code", language="python")

    eval_btn.click(
        fn=run_evaluator_optimizer,
        inputs=[eval_task, eval_reqs, eval_iter],
        outputs=[eval_trace, eval_code]
    )
    gr.Examples(
        examples=[[
            "Write a Python function to find the top N most frequent words in a text string",
            "- Signature: top_n_words(text: str, n: int) -> list[tuple[str, int]]\n- Returns (word, count) tuples sorted by frequency descending\n- Case-insensitive, strips punctuation\n- Handles empty string and n > unique words\n- stdlib only",
            3
        ]],
        inputs=[eval_task, eval_reqs, eval_iter]
    )

evaluator_demo.launch()

---
## Workflows vs. Agents — What's the Difference?

Before moving to Pattern 6, it's worth pausing on the key distinction Anthropic draws.

### Workflows
Patterns 1–5 are all **workflows** — the *developer* defines the control flow upfront:

| Pattern | Who decides the flow? |
|---|---|
| Prompt Chaining | Developer hardcodes: Step 1 → Step 2 → Step 3 |
| Routing | Developer defines the categories and handlers |
| Parallelization | Developer defines which tasks run in parallel |
| Orchestrator-Workers | Orchestrator LLM plans, but *within a fixed pipeline shape* |
| Evaluator-Optimizer | Developer defines the loop: generate → evaluate → repeat |

The LLM fills in the *content* at each step, but the *structure* is fixed in code.

```
Workflow:  [fixed step 1] → [fixed step 2] → [fixed step 3]
                ↑                 ↑                 ↑
           LLM fills         LLM fills         LLM fills
           the content       the content       the content
```

**Advantages:** Predictable, easy to test, easy to debug, cheaper to run.

---

### Agents
**Agents** let the LLM decide the flow itself — what to do next, which tools to call, when to stop.

```
Agent:   Goal → [LLM decides] → action → observe → [LLM decides] → action → ...
                      ↑                                    ↑
               picks the next step                  adapts based on result
```

**Advantages:** Handles open-ended problems, adapts to unexpected results, no need to anticipate every path.

**Trade-offs:**

| | Workflow | Agent |
|---|---|---|
| **Predictability** | High — same structure every run | Lower — path varies with input |
| **Debuggability** | Easy — fixed steps to inspect | Harder — dynamic flow |
| **Cost** | Lower — known number of LLM calls | Higher — variable, can spiral |
| **Flexibility** | Low — can't handle surprises | High — adapts at runtime |
| **Best for** | Well-defined, repeatable tasks | Open-ended, exploratory tasks |

> **Rule of thumb (Anthropic):** Start with the simplest workflow that works. Only reach for a fully autonomous agent when the task genuinely cannot be structured upfront.

---
## Pattern 6 — Autonomous Agent

**Concept:** The LLM *dynamically* directs its own process — it decides which tools to use, in what order, based on what it observes. No fixed flow.

```
Goal ──► [LLM] ──► pick tool ──► execute ──► observe result
              ▲                                     │
              └──────────── loop ◄──────────────────┘
                                    │ (done?)
                                    ▼
                               Final Answer
```

**When to use:** Open-ended problems where the solution path cannot be defined upfront.

In [ ]:
# ── Autonomous Agent: Sales Data Analysis ────────────────────────────────────
SALES_DATA = [
    {"month": "Jan", "product": "Widget A", "sales": 1200, "region": "North"},
    {"month": "Jan", "product": "Widget B", "sales": 800,  "region": "South"},
    {"month": "Feb", "product": "Widget A", "sales": 1500, "region": "North"},
    {"month": "Feb", "product": "Widget B", "sales": 950,  "region": "South"},
    {"month": "Mar", "product": "Widget A", "sales": 1100, "region": "North"},
    {"month": "Mar", "product": "Widget B", "sales": 1200, "region": "South"},
    {"month": "Apr", "product": "Widget A", "sales": 1800, "region": "North"},
    {"month": "Apr", "product": "Widget B", "sales": 750,  "region": "South"},
]

def get_data_schema() -> str:
    return json.dumps({"columns": list(SALES_DATA[0].keys()), "row_count": len(SALES_DATA), "sample": SALES_DATA[:2]})

def filter_data(column: str, value: str) -> str:
    return json.dumps([r for r in SALES_DATA if str(r.get(column, "")) == str(value)])

def aggregate_sum(column: str, group_by: str) -> str:
    groups: dict = {}
    for r in SALES_DATA:
        k = r.get(group_by, "Unknown")
        groups[k] = groups.get(k, 0) + r.get(column, 0)
    return json.dumps(groups)

def calculate_stats(values: str) -> str:
    nums = json.loads(values)
    if not nums:
        return json.dumps({"error": "empty"})
    return json.dumps({"mean": round(sum(nums)/len(nums), 2), "min": min(nums), "max": max(nums), "total": sum(nums)})

DATA_TOOL_MAP = {"get_data_schema": get_data_schema, "filter_data": filter_data,
                 "aggregate_sum": aggregate_sum, "calculate_stats": calculate_stats}

data_tool_defs = [
    {"type": "function", "function": {"name": "get_data_schema",
        "description": "Get schema and sample rows of the sales dataset",
        "parameters": {"type": "object", "properties": {}, "additionalProperties": False}}},
    {"type": "function", "function": {"name": "filter_data",
        "description": "Filter rows where column equals value",
        "parameters": {"type": "object", "properties": {
            "column": {"type": "string"}, "value": {"type": "string"}},
            "required": ["column", "value"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "aggregate_sum",
        "description": "Sum a numeric column grouped by another column",
        "parameters": {"type": "object", "properties": {
            "column": {"type": "string"}, "group_by": {"type": "string"}},
            "required": ["column", "group_by"], "additionalProperties": False}}},
    {"type": "function", "function": {"name": "calculate_stats",
        "description": "Compute mean/min/max/total for a JSON list of numbers",
        "parameters": {"type": "object", "properties": {
            "values": {"type": "string"}},
            "required": ["values"], "additionalProperties": False}}},
]

def run_data_agent(goal: str) -> tuple[str, str]:
    messages = [
        {"role": "system", "content": (
            "You are a data analysis agent. Use tools to explore the dataset and answer the question. "
            "Think step by step. When you have the complete answer, provide it."
        )},
        {"role": "user", "content": goal},
    ]
    trace_lines = []

    for step in range(1, 12):
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=data_tool_defs)
        msg    = response.choices[0].message
        finish = response.choices[0].finish_reason

        if finish == "stop":
            trace_lines.append(f"**Done** in {step} step(s).")
            return "\n".join(trace_lines), msg.content

        if finish == "tool_calls":
            messages.append(msg)
            for tc in msg.tool_calls:
                fn   = tc.function.name
                args = json.loads(tc.function.arguments)
                result = DATA_TOOL_MAP[fn](**args)
                short  = result[:100] + "..." if len(result) > 100 else result
                trace_lines.append(f"**Step {step}:** `{fn}({args})` → `{short}`")
                messages.append({"role": "tool", "content": result, "tool_call_id": tc.id})

    return "\n".join(trace_lines), "Max steps reached."

# ── Gradio UI ────────────────────────────────────────────────────────────────
with gr.Blocks(title="Autonomous Agent") as autonomous_demo:
    gr.Markdown(
        "## Pattern 6 — Autonomous Agent\n"
        "Ask any question about the sales dataset. The agent decides which tools to call and in what order — no fixed flow.\n\n"
        "**Dataset:** Widget A & B sales across Jan–Apr in North/South regions."
    )
    with gr.Row():
        with gr.Column():
            agent_input = gr.Textbox(label="Your Question", placeholder="Ask about the sales data...", lines=2)
            agent_btn   = gr.Button("Run Agent", variant="primary")
        with gr.Column():
            agent_trace  = gr.Markdown(label="Agent Tool Trace")
            agent_answer = gr.Markdown(label="Final Answer")

    agent_btn.click(fn=run_data_agent, inputs=agent_input, outputs=[agent_trace, agent_answer])
    gr.Examples(
        examples=[
            ["Which product has higher total sales overall?"],
            ["Which month had the biggest sales difference between the two products?"],
            ["What is the average monthly sales for Widget A?"],
            ["Compare total sales by region."],
        ],
        inputs=agent_input
    )

autonomous_demo.launch()

---
# Summary

## Pattern Decision Guide

```
Is the task decomposable into steps?
│
├─ YES, fixed order ─────────────────────── PROMPT CHAINING
│
├─ YES, input type determines path ─────── ROUTING
│
├─ YES, steps are independent ──────────── PARALLELIZATION
│   └─ Need higher confidence? ─────────── use VOTING variant
│
├─ YES, but steps unknown upfront ──────── ORCHESTRATOR-WORKERS
│
├─ Output quality needs iteration ──────── EVALUATOR-OPTIMIZER
│
└─ Open-ended, can't define steps ──────── AUTONOMOUS AGENT
```

## Key Principles (from Anthropic)

1. **Start simple** — use the simplest pattern that solves the problem
2. **Transparency** — make agent reasoning visible for debugging
3. **Human-in-the-loop** — for irreversible or high-stakes actions
4. **Clear stopping conditions** — agents need to know when they are done
5. **Prefer workflows over agents** when structure is known — more predictable, easier to test

## Framework Quick Reference

| Use Case | Recommended |
|---|---|
| Production OpenAI agents | OpenAI Agents SDK |
| RAG + complex pipelines | LangChain / LlamaIndex |
| Multi-agent collaboration | AutoGen / CrewAI |
| Enterprise (.NET/Azure) | Semantic Kernel |
| Claude-native + MCP | Claude Agent SDK |

---
# Part 4 — Model Context Protocol (MCP)

## What is MCP?

**MCP (Model Context Protocol)** is an open standard introduced by Anthropic that defines how AI agents communicate with external tools and data sources.

```
Without MCP (custom per-integration):          With MCP (standard protocol):
─────────────────────────────────────          ─────────────────────────────
Agent ──► custom code ──► Database             Agent
Agent ──► custom code ──► File system            └──► MCP Client
Agent ──► custom code ──► API                         └──► MCP Server ──► Database
Agent ──► custom code ──► ...                                           ──► File system
                                                                        ──► API
                                                                        ──► ...
```

## Key Concepts

| Concept | Description |
|---|---|
| **MCP Server** | Exposes tools, resources, and prompts over the protocol |
| **MCP Client** | Connects to one or more servers, discovers and calls tools |
| **Transport** | How client and server communicate — `stdio` (subprocess) or `SSE` (HTTP) |
| **Tool** | A callable function the LLM can invoke via the server |
| **Resource** | Read-only data the server can expose (files, DB records) |

## Architecture

```
┌─────────────────────────────────────────────────────────┐
│                    HOST PROCESS (notebook)               │
│                                                         │
│  ┌─────────────┐    MCP Protocol    ┌────────────────┐  │
│  │  MCP Client │ ◄────────────────► │   MCP Server   │  │
│  │  (agent)    │   list_tools()     │  (subprocess)  │  │
│  │             │   call_tool()      │                │  │
│  └─────────────┘                   │  - list_equip  │  │
│        │                           │  - get_maint   │  │
│        ▼                           │  - get_details │  │
│    OpenAI LLM                      │  - stats       │  │
│                                    └───────┬────────┘  │
└────────────────────────────────────────────┼────────────┘
                                             │
                                      maintenance.db
```

## Example 1 — The MCP Server (`mcp_server.py`)

The server is a **separate Python file** that exposes 4 tools over stdio transport.  
Run it standalone with: `python mcp_server.py`

In [ ]:
# Display the MCP server source so attendees can read it in the notebook
with open("mcp_server.py") as f:
    server_source = f.read()

from IPython.display import display
from IPython.display import Code as IPCode
display(IPCode(server_source, language="python"))

## Example 2 — Discover Tools from the MCP Server

Before building the agent, connect to the server and list what tools it exposes.  
This is how any MCP client discovers capabilities at runtime.

In [ ]:
import asyncio
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PARAMS = StdioServerParameters(
    command=sys.executable,          # same Python interpreter as the notebook
    args=["mcp_server.py"],
    env=None
)

async def list_mcp_tools() -> list:
    """Connect to the MCP server and return its tool list."""
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools_result = await session.list_tools()
            return tools_result.tools

tools_list = asyncio.run(list_mcp_tools())

print(f"MCP server exposes {len(tools_list)} tools:\n")
for t in tools_list:
    print(f"  • {t.name}")
    print(f"    {t.description}")
    print()

## Example 3 — MCP-Powered Agent with Gradio UI

Now wire the MCP server into an OpenAI agent loop.  
The agent:
1. Launches the MCP server as a subprocess
2. Discovers tools dynamically — converts MCP tool schemas to OpenAI tool format
3. Runs the standard agent loop — when the LLM requests a tool, the call goes to the MCP server
4. Returns the final answer

```
User question
     │
     ▼
[OpenAI LLM] ──── tool_call ────► [MCP Client] ──► [MCP Server subprocess]
     ▲                                                      │
     └──────────────── tool_result ◄───────────────────────┘
```

In [ ]:
def mcp_tool_schema_to_openai(mcp_tool) -> dict:
    """Convert an MCP tool definition into OpenAI function-calling format."""
    schema = mcp_tool.inputSchema if mcp_tool.inputSchema else {"type": "object", "properties": {}}
    schema.setdefault("additionalProperties", False)
    return {
        "type": "function",
        "function": {
            "name": mcp_tool.name,
            "description": mcp_tool.description or "",
            "parameters": schema,
        }
    }

async def run_mcp_agent(user_question: str) -> tuple[str, str]:
    """
    Full agent loop that talks to the MCP server for every tool call.
    Returns (trace_markdown, final_answer).
    """
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ── 1. Discover tools from MCP server ────────────────────────────
            mcp_tools   = (await session.list_tools()).tools
            openai_tools = [mcp_tool_schema_to_openai(t) for t in mcp_tools]

            # ── 2. Build initial messages ─────────────────────────────────────
            messages = [
                {"role": "system", "content": (
                    "You are a maintenance assistant for a manufacturing plant. "
                    "Use the available tools to answer questions about equipment and maintenance schedules. "
                    "Be concise and accurate."
                )},
                {"role": "user", "content": user_question},
            ]

            trace_lines = []

            # ── 3. Agent loop ─────────────────────────────────────────────────
            for step in range(1, 10):
                response = client.chat.completions.create(
                    model=MODEL, messages=messages, tools=openai_tools
                )
                msg    = response.choices[0].message
                finish = response.choices[0].finish_reason

                if finish == "stop":
                    trace_lines.append(f"**Done** in {step} step(s).")
                    return "\n".join(trace_lines), msg.content

                if finish == "tool_calls":
                    messages.append(msg)
                    for tc in msg.tool_calls:
                        fn_name = tc.function.name
                        fn_args = json.loads(tc.function.arguments)

                        # ── Call the MCP server ───────────────────────────────
                        mcp_result = await session.call_tool(fn_name, fn_args)
                        result_text = mcp_result.content[0].text if mcp_result.content else "{}"

                        short = result_text[:120] + "..." if len(result_text) > 120 else result_text
                        trace_lines.append(f"**Step {step}:** `{fn_name}({fn_args})` → `{short}`")

                        messages.append({
                            "role": "tool",
                            "content": result_text,
                            "tool_call_id": tc.id
                        })

            return "\n".join(trace_lines), "Max steps reached."


def run_mcp_agent_sync(question: str) -> tuple[str, str]:
    """Sync wrapper so Gradio can call the async agent."""
    return asyncio.run(run_mcp_agent(question))


# ── Gradio UI ────────────────────────────────────────────────────────────────
with gr.Blocks(title="MCP Agent") as mcp_demo:
    gr.Markdown(
        "## MCP-Powered Maintenance Agent\n"
        "Ask questions about the manufacturing plant's equipment and maintenance schedule.\n"
        "Every tool call goes through the **MCP server** running as a subprocess.\n\n"
        "**Available MCP tools:** `list_equipment` · `get_upcoming_maintenance` · "
        "`get_equipment_details` · `maintenance_stats_by_location`"
    )
    with gr.Row():
        with gr.Column():
            mcp_input = gr.Textbox(
                label="Question",
                placeholder="e.g. Which equipment in Plant A needs maintenance soon?",
                lines=2
            )
            mcp_btn = gr.Button("Ask Agent", variant="primary")
        with gr.Column():
            mcp_trace  = gr.Markdown(label="MCP Tool Call Trace")
            mcp_answer = gr.Markdown(label="Answer")

    mcp_btn.click(fn=run_mcp_agent_sync, inputs=mcp_input, outputs=[mcp_trace, mcp_answer])
    gr.Examples(
        examples=[
            ["Which equipment in Plant A needs maintenance in the next 30 days?"],
            ["Give me a summary of overdue maintenance by location."],
            ["What are the full details for Fan-3?"],
            ["List all equipment in the Warehouse."],
            ["Which location has the most overdue maintenance items?"],
        ],
        inputs=mcp_input
    )

mcp_demo.launch()